# Map back airway sputum annotations to the combined object

Imports

In [8]:
# %% Libraries

import os
import scanpy as sc
import numpy as np
from pathlib import Path
import rapids_singlecell as rsc
import pandas as pd
from scipy.sparse import issparse

# %% Setting Paths
MAIN_DIR_NAME = "proseg_data"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME
os.chdir(MAIN_DIR)

# %% Setting Seed
SEED_VALUE = 42
# set NumPy RNG for consistency
np.random.seed(SEED_VALUE)

# %% object versions
COMB_V= 'refined-clustered'
COMB_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb-{COMB_V}.h5ad'

NEW_COMB_V = 'refined-clustered'
NEW_COMB_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb-{NEW_COMB_V}.h5ad'

SUB_OBJ = "B"
SUB_OBJ_V = 'reclustered'
SUB_OBJ_PATH = MAIN_DIR / 'data' / 'sub' / SUB_OBJ / 'h5ad' / f'{SUB_OBJ}-{SUB_OBJ_V}.h5ad'

In [2]:
# load comb obj
comb = sc.read_h5ad(COMB_PATH)

In [ ]:
# load airway epi obj
sub = sc.read_h5ad(SUB_OBJ_PATH)

Get all the cell names and annotations

In [ ]:
sub_ann = sub.obs[["leiden","reclustered_ct"]].copy()

In [ ]:

sub_ann = sub_ann.rename(columns={
    "leiden": "plasma_leiden_n10_0.3",
    "reclustered_ct": "ann_lvl_3_refined",
})

sub_ann

Map back

In [ ]:
# Add the airway Leiden clusters; non-airway cells will be NaN
comb.obs["plasma_leiden_n10_0.3"] = (
    sub_ann["plasma_leiden_n10_0.3"]
    .reindex(comb.obs.index)
)

# Start with the original annotation for every cell
comb.obs["ann_lvl_3_refined"] = comb.obs["ann_lvl_3_refined"].astype(str)

# Replace annotations only for cells present in sub_ann
common_cells = comb.obs.index.intersection(sub_ann.index)

comb.obs.loc[common_cells, "ann_lvl_3_refined"] = (
    sub_ann.loc[common_cells, "ann_lvl_3_refined"].astype(str)
)

In [ ]:
sc.pl.umap(
    comb, 
    color=["ann_lvl_3_refined"],
    #groups=['Amb'],
    legend_loc="on data"
)

Save the annotated object

In [18]:
comb.write_h5ad(NEW_COMB_PATH)